In [1]:
import qewton

In [2]:
X = qewton.Variable("x", 3)
U = qewton.Variable("u", 3)
P = qewton.Variable("p", 1)

In [3]:
mesh_geo = qewton.geometries.MeshGeometry.load_mesh(
    variable=X, file_path="data/stokes_pipe/u_pipe.msh"
)
inflow_goe = mesh_geo.boundary.get_submesh(1)
outflow_geo = mesh_geo.boundary.get_submesh(2)
side_geo = mesh_geo.boundary.get_submesh(3)

/home/tomfre/Desktop/pioneer-backend/src/qewton/geometries/discrete/mesh.py:121: RuntimeWarning: invalid value encountered in divide
  self.boundary_normals_at_vertex /= np.linalg.norm(


In [4]:
volume_sampler = qewton.RandomUniformSampler(mesh_geo, 10000)
inflow_sampler = qewton.RandomUniformSampler(inflow_goe, 1000)
outflow_sampler = qewton.RandomUniformSampler(outflow_geo, 1000)
side_sampler = qewton.RandomUniformSampler(side_geo, 3000)

In [5]:
# import matplotlib.pyplot as plt

# in_points = inflow_sampler()
# out_points = outflow_sampler()
# side_points = side_sampler()
# # Create figure
# fig = plt.figure(figsize=(8, 6))

# # Add 3D axes
# ax = fig.add_subplot(111, projection='3d')

# # Scatter plot
# ax.scatter(in_points[:, 0], in_points[:, 1], in_points[:, 2])
# ax.scatter(out_points[:, 0], out_points[:, 1], out_points[:, 2])
# ax.scatter(side_points[:, 0], side_points[:, 1], side_points[:, 2])

# # Axis labels
# ax.set_xlabel("X")
# ax.set_ylabel("Y")
# ax.set_zlabel("Z")

# plt.show()

In [6]:
model = qewton.FCN(
    in_neurons=X,
    hidden_neurons=50,
    out_neurons=U+P,
    n_hidden_layers=5,
    activation=qewton.bb.Tanh,
)

In [8]:
def momentum_residual(u: U, p: P, x: X):  # type: ignore
    return u.sym_grad(x).matrix_div(x) - p.gradient(x)

momentum_graph = qewton.PINNPipeline(volume_sampler, [model], 
                                     residual=momentum_residual, 
                                     residual_name="MomentumConstraint")
momentum_constraint = momentum_graph.constraint

In [9]:
def mass_residual(u: U, x: X):  # type: ignore
    return u.div(x)

mass_graph = qewton.PINNPipeline(volume_sampler, [model], 
                                 residual=mass_residual, 
                                 residual_name="MassConstraint")
mass_constraint = mass_graph.constraint

In [10]:
def inflow_residual(p: P):  # type: ignore
    return p - 1.0

in_graph = qewton.PINNPipeline(inflow_sampler, [model], 
                                residual=inflow_residual, 
                                residual_name="InConstraint")
in_constraint = in_graph.constraint